In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

In [6]:
train_dir = "../data/hsrp_dataset/train"
val_dir = "../data/hsrp_dataset/val"

batch_size = 16
epochs = 20
learning_rate = 0.0003

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [7]:
transform = transforms.Compose([
    
    transforms.Resize((224,224)),
    
    transforms.RandomHorizontalFlip(),
    
    transforms.RandomRotation(10),
    
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor()
    
])

In [8]:
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

class_names = train_dataset.classes
print("Classes:", class_names)

Classes: ['hsrp', 'non-hsrp']


In [9]:
model = models.mobilenet_v3_small(pretrained=True)

# Freeze feature extractor
for param in model.features.parameters():
    param.requires_grad = False

# Modify classifier
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)

model = model.to(device)

In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [11]:
train_losses = []

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}")

Epoch 1/20 - Loss: 0.6285
Epoch 2/20 - Loss: 0.5152
Epoch 3/20 - Loss: 0.4857
Epoch 4/20 - Loss: 0.4510
Epoch 5/20 - Loss: 0.4104
Epoch 6/20 - Loss: 0.3901
Epoch 7/20 - Loss: 0.3903
Epoch 8/20 - Loss: 0.3419
Epoch 9/20 - Loss: 0.3428
Epoch 10/20 - Loss: 0.3046
Epoch 11/20 - Loss: 0.2940
Epoch 12/20 - Loss: 0.3017
Epoch 13/20 - Loss: 0.2745
Epoch 14/20 - Loss: 0.2598
Epoch 15/20 - Loss: 0.2787
Epoch 16/20 - Loss: 0.2225
Epoch 17/20 - Loss: 0.2281
Epoch 18/20 - Loss: 0.2239
Epoch 19/20 - Loss: 0.2043
Epoch 20/20 - Loss: 0.1966


In [12]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    
    for images, labels in val_loader:
        
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Validation Accuracy:", accuracy)

Validation Accuracy: 91.5


In [13]:
import os
os.makedirs("../models", exist_ok=True)

torch.save(model.state_dict(), "../models/hsrp_classifier.pth")

print("Model saved to ../models/hsrp_classifier.pth")

Model saved to ../models/hsrp_classifier.pth
